# GSSS_014 - Prompt Engineering, by API (not ChatGPT)

We use the **API**, not the ChatGPT app, because prompt engineering is about *controlling*
the model - and the app hides the controls (temperature, the raw message list, which model
you're on) and quietly "fixes" a weak prompt so you never see it fail.

Every demo runs the **same prompt on three Groq models** at `temperature=0`:

| key | model | why it's here |
|---|---|---|
| `qwen` | `qwen/qwen3.8-27b` | strong instruction-follower |
| `oss-20b` | `openai/gpt-oss-20b` | small - benefits most from good prompting |
| `oss-120b` | `openai/gpt-oss-120b` | large - the "does it still matter?" check |

A weak prompt gives you **three different answers**; a good prompt gives you **three
answers that agree**. That agreement is the signal your prompt is doing the work, not the
model's guesswork.

**Techniques covered:** RCGCOS structure · zero/one/few-shot · chain-of-thought ·
tree-of-thought · negative prompting · self-verification · ReAct (tools) · self-consistency ·
structured (JSON) output. Every one is shown **with vs without**, and all the example
prompts are collected at the end for copy-paste.


## Section 0 - Setup

**Run this notebook from top to bottom.** The first code cell installs the packages into the active notebook Python environment, which avoids the common `ModuleNotFoundError: langchain_groq` problem.

Do not paste your API key into the notebook source. The setup cell asks for it securely.


In [2]:
import sys, subprocess

# Install into the SAME Python environment used by this notebook.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "langchain", "langchain-groq", "ddgs", "gradio", "requests"
])
print("Dependencies installed successfully.")


Dependencies installed successfully.


In [9]:
import os, re, json, time
from getpass import getpass

def ask_key(name):
    if os.getenv(name):
        print(f"{name}: from environment"); return os.environ[name]
    return getpass(f"Paste {name}: ").strip()

GROQ_API_KEY = ask_key("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

from langchain_groq import ChatGroq

def _mk(model, temperature=0):
    # 2000 tokens: gpt-oss models "think" first and can run out before answering on a short budget
    return ChatGroq(model=model, api_key=GROQ_API_KEY, temperature=temperature,
                    max_tokens=800, max_retries=2, timeout=60)

MODELS = {
    "qwen":     _mk("qwen/qwen3.8-27b"),
    "oss-20b":  _mk("openai/gpt-oss-20b"),
    "oss-120b": _mk("openai/gpt-oss-120b"),
}

def _clean(t):
    return re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip()   # hide reasoning traces

def ask(prompt, model="qwen", raw=False, **kw):
    """One model, one prompt."""
    out = MODELS[model].invoke(prompt, **kw).content
    return out if raw else _clean(out)

def compare(prompt, models=None, width=1300):
    """Same prompt on every model, printed side by side."""
    models = models or list(MODELS)
    print("PROMPT:\n" + prompt.strip() + "\n" + "-" * 74)
    for k in models:
        try:
            out = _clean(MODELS[k].invoke(prompt).content)
        except Exception as e:
            out = f"<error: {e}>"
        print(f"\n### {k}\n{out[:width]}")
    print()

GALLERY = []
def demo(title, without_prompt, with_prompt, run=True):
    """Register the pair for the end-of-notebook gallery, and (optionally) run both now."""
    GALLERY.append((title, without_prompt.strip(), with_prompt.strip()))
    if run:
        print("=" * 74 + f"\n{title}\n" + "=" * 74)
        print("\n>>>>>>>>>> WITHOUT the technique\n")
        compare(without_prompt)
        print(">>>>>>>>>> WITH the technique\n")
        compare(with_prompt)
    return without_prompt, with_prompt

print("models:", list(MODELS))
print(ask("Reply with one word: ready"))

GROQ_API_KEY: from environment
models: ['qwen', 'oss-20b', 'oss-120b']
Ready


## Section 0.1 - Warm-up: one prompt, three models

A vague ask - each model fills the gaps differently (length, angle, format). Then the same
request with a little structure - the three line up.

In [4]:
print("VAGUE\n")
compare("Tell me about transformers.")

print("\nSTRUCTURED\n")
compare("In exactly 3 bullet points, explain the Transformer neural-network architecture "
        "to a 2nd-year engineering student. Each bullet <= 20 words.")

VAGUE

PROMPT:
Tell me about transformers.
--------------------------------------------------------------------------

### qwen
<error: Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.8-27b` in organization `org_01m1dh8gwaemyaw312v5r6b6m8` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1081. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing", 'type': 'tokens', 'code': 'rate_limit_exceeded'}}>

### oss-20b
Sure! Just to make sure I give you the information you’re looking for, could you let me know which type of transformers you’re interested in?

- **Transformer neural networks** (the architecture behind models like GPT, BERT, etc.)  
- **Electrical transformers** (devices that step voltage up or down in power systems)  

Let me know which one (or bot

## Section 1 - RCGCOS: give the prompt a skeleton

Six slots. Fill the ones that matter for your task:

| slot | question it answers |
|---|---|
| **R**ole | who is the model being right now? |
| **C**ontext | what background does it need? |
| **G**oal | what exactly do you want produced? |
| **C**onstraints | length, what to avoid, what must be true |
| **O**utput format | prose / table / JSON / bullets / code |
| **S**tyle | tone, reading level, person |


In [5]:
W, _ = demo(
 "RCGCOS structure",
 # WITHOUT
 "Write something about our new campus bus tracking app.",
 # WITH
 """ROLE: You are a product marketing writer for a college tech team.
CONTEXT: We built 'BusWhere', a web app showing live campus-shuttle locations and ETAs,
used by ~4000 students. Launch is next week.
GOAL: Write the app-store short description.
CONSTRAINTS: 45-60 words. No exclamation marks. Do not use the words revolutionary,
seamless, or game-changer. One concrete benefit, not three.
OUTPUT FORMAT: a single paragraph.
STYLE: plain, factual, for students.""")

RCGCOS structure

>>>>>>>>>> WITHOUT the technique

PROMPT:
Write something about our new campus bus tracking app.
--------------------------------------------------------------------------

### qwen
Here are a few different ways to write about your new campus bus tracking app, depending on where you plan to publish it (social media, a newsletter, or a press release).

### Option 1: Social Media Post (Instagram/LinkedIn/Twitter)
**Tone:** Exciting, concise, and student-focused.

**Headline:** Never miss your bus again! 🚌📱

**Body:**
Say goodbye to standing in the rain wondering when the next shuttle is coming. We’re thrilled to introduce **[App Name]**, our brand-new real-time campus bus tracking app!

✅ **Live GPS Tracking:** See exactly where your bus is right now.
✅ **Accurate ETAs:** Know precisely when it will arrive at your stop.
✅ **Route Alerts:** Get notified when your bus is 2 minutes away.

Download it now on the App Store and Google Play, and start your day with less stress

## Section 2 - Zero-shot vs one-shot vs few-shot

The label is easy; the **output shape** is where zero-shot drifts - each model picks its
own verbosity and layout. A few labelled examples pin the exact format across all three.

In [6]:
T = """T1: I was charged twice for March.
T2: The app crashes when I tap Export.
T3: Where do I change my email address?
T4: Refund never showed up on my card.
T5: How do I share a folder with my team?"""

zero = ("Classify each support ticket as BUG, BILLING, or HOW-TO, and give a short reason.\n" + T)

few = ("""Classify each support ticket. Output one line per ticket in EXACTLY this format:
<id> | <BUG|BILLING|HOWTO> | <reason in 5 words or fewer>

Examples:
X1 | BUG | crashes when opening settings
X2 | BILLING | double charged last month
X3 | HOWTO | wants to rename a project

Now do these:
""" + T)

demo("Few-shot (format lock)", zero, few)

Few-shot (format lock)

>>>>>>>>>> WITHOUT the technique

PROMPT:
Classify each support ticket as BUG, BILLING, or HOW-TO, and give a short reason.
T1: I was charged twice for March.
T2: The app crashes when I tap Export.
T3: Where do I change my email address?
T4: Refund never showed up on my card.
T5: How do I share a folder with my team?
--------------------------------------------------------------------------

### qwen
<error: Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.8-27b` in organization `org_01m1dh8gwaemyaw312v5r6b6m8` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1064. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing", 'type': 'tokens', 'code': 'rate_limit_exceeded'}}>

### oss-20b
**T1:** BILLING – “charged twice” indicates a billi

('Classify each support ticket as BUG, BILLING, or HOW-TO, and give a short reason.\nT1: I was charged twice for March.\nT2: The app crashes when I tap Export.\nT3: Where do I change my email address?\nT4: Refund never showed up on my card.\nT5: How do I share a folder with my team?',
 'Classify each support ticket. Output one line per ticket in EXACTLY this format:\n<id> | <BUG|BILLING|HOWTO> | <reason in 5 words or fewer>\n\nExamples:\nX1 | BUG | crashes when opening settings\nX2 | BILLING | double charged last month\nX3 | HOWTO | wants to rename a project\n\nNow do these:\nT1: I was charged twice for March.\nT2: The app crashes when I tap Export.\nT3: Where do I change my email address?\nT4: Refund never showed up on my card.\nT5: How do I share a folder with my team?')

## Section 3 - Chain of Thought (CoT)

Ask for the reasoning *before* the answer. It raises accuracy on multi-step problems and -
just as important - makes the answer **auditable**: you can see where it went wrong.

> Modern "reasoning" models (qwen, gpt-oss) do some of this internally, so the gap is
> smaller for them than for older/smaller models - but explicit CoT still makes the working
> visible and the format controllable.

In [7]:
puzzle = ("A shop sells pens at 3 for Rs 20 and notebooks at Rs 45 each. "
          "Ravi buys 12 pens and 2 notebooks, pays with a Rs 200 note. "
          "How much change does he get?")

demo("Chain of Thought",
     puzzle + " Give only the final number.",
     puzzle + " Think step by step: pens cost, notebooks cost, total, then change. "
              "Show each step, then give the final number on its own line.")

Chain of Thought

>>>>>>>>>> WITHOUT the technique

PROMPT:
A shop sells pens at 3 for Rs 20 and notebooks at Rs 45 each. Ravi buys 12 pens and 2 notebooks, pays with a Rs 200 note. How much change does he get? Give only the final number.
--------------------------------------------------------------------------

### qwen
1

### oss-20b
30

### oss-120b
30

>>>>>>>>>> WITH the technique

PROMPT:
A shop sells pens at 3 for Rs 20 and notebooks at Rs 45 each. Ravi buys 12 pens and 2 notebooks, pays with a Rs 200 note. How much change does he get? Think step by step: pens cost, notebooks cost, total, then change. Show each step, then give the final number on its own line.
--------------------------------------------------------------------------

### qwen
<error: Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.8-27b` in organization `org_01m1dh8gwaemyaw312v5r6b6m8` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1064. The 

('A shop sells pens at 3 for Rs 20 and notebooks at Rs 45 each. Ravi buys 12 pens and 2 notebooks, pays with a Rs 200 note. How much change does he get? Give only the final number.',
 'A shop sells pens at 3 for Rs 20 and notebooks at Rs 45 each. Ravi buys 12 pens and 2 notebooks, pays with a Rs 200 note. How much change does he get? Think step by step: pens cost, notebooks cost, total, then change. Show each step, then give the final number on its own line.')

## Section 4 - Tree of Thought (ToT)

Instead of one reasoning path, make the model **branch**: generate a few distinct
strategies, judge them, then expand the best one. Helps on problems where the first
approach is often the wrong one.

In [10]:
budget = ("You have Rs 100. Items cost Rs 18, 23, 31, 44, 55. Pick a set of items whose "
          "total is as close to Rs 100 as possible WITHOUT going over. Which items, and what total?")

def tree_of_thought(problem):
    strategies = ask(f"{problem}\nList 3 genuinely different strategies to find the answer. "
                     f"Number them, one line each.")
    scored = ask(f"Problem: {problem}\n\nStrategies:\n{strategies}\n\n"
                 f"Rate each strategy 1-10 for how reliably it finds the exact best answer. "
                 f"Say which number is best.")
    final = ask(f"Problem: {problem}\n\nUse this strategy:\n{scored}\n\n"
                f"Now carry it out fully - check the promising combinations - and state the "
                f"winning set and its total.")
    return strategies, scored, final

demo("Tree of Thought (framing)",
     budget + " Answer directly.",
     budget + " Consider several combinations before answering; do not just add the biggest items.",
     run=True)

print("\n\n----- explicit ToT orchestration (qwen) -----")
s, sc, f = tree_of_thought(budget)
print("BRANCHES:\n", s, "\n\nJUDGED:\n", sc, "\n\nRESULT:\n", f)

Tree of Thought (framing)

>>>>>>>>>> WITHOUT the technique

PROMPT:
You have Rs 100. Items cost Rs 18, 23, 31, 44, 55. Pick a set of items whose total is as close to Rs 100 as possible WITHOUT going over. Which items, and what total? Answer directly.
--------------------------------------------------------------------------

### qwen
Items: 55, 44
Total: Rs 99

### oss-20b
Items: Rs 44 and Rs 55  
Total: Rs 99 (the closest to Rs 100 without exceeding it)

### oss-120b
**Items:** Rs 55 + Rs 44  
**Total:** Rs 99 (the closest possible to Rs 100 without exceeding it).

>>>>>>>>>> WITH the technique

PROMPT:
You have Rs 100. Items cost Rs 18, 23, 31, 44, 55. Pick a set of items whose total is as close to Rs 100 as possible WITHOUT going over. Which items, and what total? Consider several combinations before answering; do not just add the biggest items.
--------------------------------------------------------------------------

### qwen
To find the set of items that sums to a total as clos

## Section 5 - Negative prompting

Telling the model what **not** to do is a real lever - explicit prohibitions are obeyed
more reliably than vague "keep it professional".

In [11]:
demo("Negative prompting",
 "Write a 3-sentence description of a reusable steel water bottle for an online store.",
 """Write a 3-sentence description of a reusable steel water bottle for an online store.
Do NOT use any of these words: eco-friendly, sustainable, revolutionary, game-changer,
premium, perfect, amazing. Do NOT use exclamation marks. Do NOT mention the environment.
State only physical facts a buyer can check.""")

Negative prompting

>>>>>>>>>> WITHOUT the technique

PROMPT:
Write a 3-sentence description of a reusable steel water bottle for an online store.
--------------------------------------------------------------------------

### qwen
Stay hydrated in style with our premium stainless steel water bottle, engineered to keep your drinks ice-cold for 24 hours or piping hot for 12. Its durable, double-walled construction ensures zero condensation and maximum protection, making it the perfect companion for your daily commute, gym sessions, or outdoor adventures. Choose from a variety of sleek colors to match your personal aesthetic while reducing single-use plastic waste.

### oss-20b
Stay hydrated on the go with our sleek, double‑walled stainless‑steel water bottle, engineered to keep drinks cold for 24 hours or hot for 12 hours. Its BPA‑free, leak‑proof cap and 20 oz capacity make it the perfect companion for workouts, hikes, or daily commutes. Built to last, the bottle’s durable construction

('Write a 3-sentence description of a reusable steel water bottle for an online store.',
 'Write a 3-sentence description of a reusable steel water bottle for an online store.\nDo NOT use any of these words: eco-friendly, sustainable, revolutionary, game-changer,\npremium, perfect, amazing. Do NOT use exclamation marks. Do NOT mention the environment.\nState only physical facts a buyer can check.')

## Section 6 - Self-verification (catching hallucination)

Models state wrong things confidently. Adding "...and if you are not sure, say so" or
"...then check your answer" recovers a lot of it - the model was capable of the right
answer, it just wasn't asked to look twice.

In [12]:
# 6a. a made-up paper - the model will happily summarise something that does not exist
demo("Self-verification (fake source)",
 "Give a two-sentence summary of the 2019 paper 'Recurrent Attention Collapse in Deep "
 "Transformers' by K. Nakamura and P. Alvarez.",
 "Give a two-sentence summary of the 2019 paper 'Recurrent Attention Collapse in Deep "
 "Transformers' by K. Nakamura and P. Alvarez. If you cannot identify this exact paper "
 "with confidence, say it does not appear to exist rather than summarising it.")

print("\n\n----- 6b. show-the-working (auditability) -----")
demo("Self-verification (show working)",
 "Is 667 a prime number? Answer yes or no only.",
 "Is 667 a prime number? Verify by trial division against every prime up to sqrt(667) "
 "(about 25.8): 2,3,5,7,11,13,17,19,23. Show each test, then answer.")

Self-verification (fake source)

>>>>>>>>>> WITHOUT the technique

PROMPT:
Give a two-sentence summary of the 2019 paper 'Recurrent Attention Collapse in Deep Transformers' by K. Nakamura and P. Alvarez.
--------------------------------------------------------------------------

### qwen
The paper "Recurrent Attention Collapse in Deep Transformers" by K. Nakamura and P. Alvarez does not exist, as there is no record of such a publication in academic databases or major machine learning conferences. Consequently, it is not possible to provide a summary of a non-existent work.

### oss-20b
The 2019 study by Nakamura and Alvarez identifies a “recurrent attention collapse” phenomenon in deep transformer architectures, where successive layers increasingly converge to a small subset of attention heads, thereby reducing model expressiveness and robustness. To counter this, the authors propose a lightweight regularization scheme that encourages diversity among attention patterns, demonstrating i

('Is 667 a prime number? Answer yes or no only.',
 'Is 667 a prime number? Verify by trial division against every prime up to sqrt(667) (about 25.8): 2,3,5,7,11,13,17,19,23. Show each test, then answer.')

## Section 7 - ReAct: reason + act (use a tool)

When the answer needs current or checkable facts, give the model a **tool** and let it
loop: Thought -> Action (search) -> Observation -> ... -> Answer. Without the tool it
guesses from stale training data.

In [13]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def web_search(query: str) -> str:
    """Search the web for current facts. Returns short snippets with links."""
    try:
        from ddgs import DDGS
        hits = list(DDGS().text(query, max_results=5))
        if hits:
            return "\n".join(f"- {h['title']}: {h['body'][:180]} <{h['href']}>" for h in hits)
    except Exception:
        pass
    try:
        import requests
        ua = {"User-Agent": "GSSS-Bot/1.0"}
        s = requests.get("https://en.wikipedia.org/w/api.php", headers=ua, timeout=15, params={
            "action": "query", "list": "search", "srsearch": query, "format": "json", "srlimit": 3}).json()
        return "\n".join(f"- {r['title']}: {re.sub('<.*?>','',r['snippet'])}" for r in s["query"]["search"])
    except Exception as e:
        return f"search failed: {e}"

QUESTION = "Which team won the most recent Cricket World Cup, and in what year?"

print(">>> WITHOUT a tool (from training data)\n")
compare(QUESTION + " Answer in one sentence.")

print("\n>>> WITH ReAct (qwen + web_search)\n")
agent = create_agent(MODELS["qwen"], [web_search],
    system_prompt="Answer using web_search when the fact may be recent. Cite what you found.")
out = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]},
                   {"recursion_limit": 8})
tools_used = [tc["name"] for m in out["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
print("tools called:", tools_used)
print(out["messages"][-1].content)

GALLERY.append(("ReAct (tool use)",
    QUESTION + "  (no tool -> stale/guessed answer)",
    QUESTION + "  (agent with web_search -> searches, then answers with a source)"))

>>> WITHOUT a tool (from training data)

PROMPT:
Which team won the most recent Cricket World Cup, and in what year? Answer in one sentence.
--------------------------------------------------------------------------

### qwen
Australia won the most recent Cricket World Cup in 2023.

### oss-20b
England won the most recent Cricket World Cup, which took place in 2023.

### oss-120b
Australia won the most recent Cricket World Cup in 2023.


>>> WITH ReAct (qwen + web_search)

tools called: ['web_search', 'web_search', 'web_search']
**Australia** won the most recent ICC Men's Cricket World Cup in **2023**.

They defeated India by 6 wickets in the final, played on November 19, 2023, at the Narendra Modi Stadium in Ahmedabad, India. This was Australia's sixth ODI World Cup title.

Sources:
- [ESPN Cricinfo – 2023 Cricket World Cup Final Scorecard](https://www.espncricinfo.com/series/icc-cricket-world-cup-2023-24-1367856/india-vs-australia-final-1384439/full-scorecard)
- [Olympics.com – Crick

## Section 8 - Two more that pay off

**Self-consistency:** run CoT several times with a little randomness, take the majority
answer. **Structured output:** demand JSON to a named shape so code can consume it.

In [14]:
# --- self-consistency: sample a few times, take the majority ---
q = ("Three friends split a Rs 1200 bill. Anita pays Rs 200 more than Bala. "
     "Bala pays Rs 100 less than Chetan. How much does Chetan pay in rupees?")
hot = _mk("qwen/qwen3.8-27b", temperature=0.8)

def _final_number(text):
    m = re.search(r"ANSWER\s*[:=]\s*(?:Rs\.?\s*)?([\d,]+)", text, re.I)
    if m:
        return m.group(1).replace(",", "")
    nums = re.findall(r"\b(\d{2,4})\b", text.replace(",", ""))   # fallback: last number in range
    nums = [n for n in nums if 100 <= int(n) <= 1200]
    return nums[-1] if nums else None

votes = []
for i in range(5):
    r = _clean(hot.invoke(q + "\nReason briefly, then end with a line exactly like: ANSWER: <number>").content)
    v = _final_number(r)
    if v:
        votes.append(v)
print("5 samples ->", votes)
print("majority  ->", max(set(votes), key=votes.count) if votes else "no parse", " (correct = 400)")

GALLERY.append(("Self-consistency", q + "  (one shot)",
    q + "  (sample the answer 5x at temperature 0.8, take the majority)"))

5 samples -> ['400', '400', '400', '400', '400']
majority  -> 400  (correct = 400)


In [15]:
# --- structured (JSON) output ---
review = "Arrived a day late and the box was dented, but the blender itself works great and is quiet."
schema = ('{"sentiment": "positive|negative|mixed", "product_ok": true/false, '
          '"delivery_issue": true/false, "one_line_summary": "<= 12 words"}')

demo("Structured output (JSON)",
 f"Summarise this customer review: {review}",
 f"From the review below, output ONLY valid JSON with this exact shape:\n{schema}\n"
 f"Review: {review}")

Structured output (JSON)

>>>>>>>>>> WITHOUT the technique

PROMPT:
Summarise this customer review: Arrived a day late and the box was dented, but the blender itself works great and is quiet.
--------------------------------------------------------------------------

### qwen
The customer received the blender a day late with a dented box, but they are satisfied with the product, noting that it works well and operates quietly.

### oss-20b
The blender works well and is quiet, but it arrived a day late and the box was dented.

### oss-120b
The delivery was delayed and the packaging was damaged, but the blender performs well and operates quietly.

>>>>>>>>>> WITH the technique

PROMPT:
From the review below, output ONLY valid JSON with this exact shape:
{"sentiment": "positive|negative|mixed", "product_ok": true/false, "delivery_issue": true/false, "one_line_summary": "<= 12 words"}
Review: Arrived a day late and the box was dented, but the blender itself works great and is quiet.
-------

('Summarise this customer review: Arrived a day late and the box was dented, but the blender itself works great and is quiet.',
 'From the review below, output ONLY valid JSON with this exact shape:\n{"sentiment": "positive|negative|mixed", "product_ok": true/false, "delivery_issue": true/false, "one_line_summary": "<= 12 words"}\nReview: Arrived a day late and the box was dented, but the blender itself works great and is quiet.')

## Section 9 - Playground

**Use the dropdown** to load a ready-made *without / with* pair into the two boxes - every
technique from this notebook, plus a few everyday tasks. Edit either box and hit Run to
see both versions on all three models. `temperature=0`, so any disagreement is the prompt's
fault, not randomness.

In [16]:
import gradio as gr

# a few extra hand-written pairs on top of the notebook's own examples (GALLERY)
EXTRAS = {
 "Everyday: email reply": (
   "Reply to this customer email: 'My order is 5 days late and nobody has replied.'",
   "ROLE: support agent for an online store.\nGOAL: reply to the email below.\n"
   "CONSTRAINTS: 60-80 words; apologise once; give ONE concrete next step; do not promise a delivery date.\n"
   "STYLE: calm, plain, no exclamation marks.\n"
   "EMAIL: 'My order is 5 days late and nobody has replied.'"),
 "Everyday: extract to table": (
   "Pull the key points from: 'The 2pm meeting moved to Thursday 4pm in Room 3. Bring the Q2 numbers. Priya is away.'",
   "From the text, output ONLY a markdown table with columns | field | value |, one row each "
   "for date, time, room, bring, absent.\n"
   "Text: 'The 2pm meeting moved to Thursday 4pm in Room 3. Bring the Q2 numbers. Priya is away.'"),
 "Everyday: explain a concept": (
   "Explain overfitting.",
   "ROLE: ML tutor. GOAL: explain overfitting to a 2nd-year student. "
   "CONSTRAINTS: exactly 3 sentences, one everyday analogy, no formulas. STYLE: plain."),
 "Everyday: rewrite for tone": (
   "Make this polite: 'You sent the wrong file again. Fix it.'",
   "Rewrite the message to be polite and professional. Keep it under 2 sentences. "
   "Do NOT add greetings or sign-offs. Keep the ask clear.\n"
   "Message: 'You sent the wrong file again. Fix it.'"),
}
PRESETS = {**{f"Technique: {t}": (wo, wi) for t, wo, wi in GALLERY}, **EXTRAS}

def _run_on_all(prompt):
    parts = []
    for k in MODELS:
        try:
            out = _clean(MODELS[k].invoke(prompt).content)[:1400]
        except Exception as e:
            out = f"<error: {e}>"
        parts.append(f"**{k}**\n\n{out}")
    return "\n\n---\n\n".join(parts)

def run_both(without_p, with_p):
    return _run_on_all(without_p), _run_on_all(with_p)

def load_preset(name):
    return PRESETS.get(name, ("", ""))

with gr.Blocks(title="GSSS_014 - Prompt Playground") as demo_ui:
    gr.Markdown("# Prompt Engineering Playground\n"
                "Pick an example below (it fills both boxes), or type your own. "
                "Then run the weak and the structured prompt on all three Groq models.")
    pick = gr.Dropdown(list(PRESETS), value=list(PRESETS)[0],
                       label="Example  ->  loads Prompt A and Prompt B")
    with gr.Row():
        wo = gr.Textbox(label="Prompt A - WITHOUT the technique", lines=8)
        wi = gr.Textbox(label="Prompt B - WITH structure / technique", lines=8)
    go = gr.Button("Run A and B on qwen + gpt-oss-20b + gpt-oss-120b", variant="primary")
    with gr.Row():
        wo_out = gr.Markdown()
        wi_out = gr.Markdown()

    demo_ui.load(load_preset, pick, [wo, wi])
    pick.change(load_preset, pick, [wo, wi])
    go.click(run_both, [wo, wi], [wo_out, wi_out])

demo_ui.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5590ed26ff199706a1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Section 10 - When to reach for what

| Technique | Use it when... | If you don't (what goes wrong) |
|---|---|---|
| **RCGCOS structure** | any task you'll reuse or hand to others; output varies run-to-run or model-to-model | model guesses role / length / tone / format -> inconsistent length, wrong register, "helps" with things you didn't ask for |
| **Zero-shot** | common, unambiguous tasks (translate, simple classify, summarise) | usually fine - move up only if the shape drifts |
| **One / few-shot** | a strict output *shape*, a custom label scheme, a subjective boundary, a fixed style | format drifts (markdown vs plain vs JSON, verbose vs terse); edge cases handled differently each run. Costs context tokens |
| **Chain of Thought** | multi-step arithmetic / logic; answer depends on intermediate results; you need to audit *why* | confident wrong answer on multi-hop problems, and you can't see where it broke. Skip for simple lookups - wastes tokens, can overthink |
| **Tree of Thought** | the obvious first approach is usually wrong (planning, optimisation, puzzles) and CoT alone still fails | commits to the first path and can't recover. Costs several LLM calls - only when correctness matters more than cost |
| **Negative prompting** | the model keeps doing something unwanted (cliches, emojis, hedging, banned words, over-length); brand / compliance rules | "keep it professional" is ignored; "do NOT use X, Y, Z" is obeyed. Too many negatives -> stiff output, so pair with "state only X" |
| **Self-verification** | factual claims, citations, calculations - anywhere a confident wrong answer is costly | it invents papers, authors, API methods, statistics and states them like real facts. Not a guarantee (can also "verify" a wrong answer) - best paired with a tool |
| **ReAct (tools)** | answer needs current info, private data, real computation, or a checkable source | answers from stale training data or guesses, with no way to tell it's wrong. Slower; needs tool infra |
| **Self-consistency** | a reasoning task with one correct answer; a single CoT run is sometimes wrong; you can afford N calls | one sample may be an outlier / arithmetic slip. Costs ~5x; useless for open-ended generation |
| **Structured output (JSON)** | the output feeds code / a database / an API; you process many items programmatically | you get human-readable prose that **code can't parse**; field names and structure vary per call; you end up writing brittle regex. The model may still emit invalid JSON (fences, trailing comma) -> add "output ONLY JSON" + a shown schema + a parse-retry |

Two more the notebook uses without a section of their own:
- **Delimiters** (triple backticks, `---`, XML-style tags) - separate your instructions from the user's data, so the model can't confuse the two and injected text in the data is inert.
- **Prompt chaining / decomposition** - split a big task into piped prompts (A extracts facts -> B writes from only those facts). Each step is simpler and separately checkable.


## Section 11 - The prompt gallery (copy-paste)

In [17]:
for i, (title, wo, wi) in enumerate(GALLERY, 1):
    print("#" * 78)
    print(f"# {i}. {title}")
    print("#" * 78)
    print("\n--- WITHOUT ---\n" + wo)
    print("\n--- WITH ---\n" + wi)
    print()

##############################################################################
# 1. Tree of Thought (framing)
##############################################################################

--- WITHOUT ---
You have Rs 100. Items cost Rs 18, 23, 31, 44, 55. Pick a set of items whose total is as close to Rs 100 as possible WITHOUT going over. Which items, and what total? Answer directly.

--- WITH ---
You have Rs 100. Items cost Rs 18, 23, 31, 44, 55. Pick a set of items whose total is as close to Rs 100 as possible WITHOUT going over. Which items, and what total? Consider several combinations before answering; do not just add the biggest items.

##############################################################################
# 2. Negative prompting
##############################################################################

--- WITHOUT ---
Write a 3-sentence description of a reusable steel water bottle for an online store.

--- WITH ---
Write a 3-sentence description of a reusable stee

## Recap

- **Structure first (RCGCOS).** Role, Context, Goal, Constraints, Output format, Style -
  fill what matters, and your three models start agreeing.
- **Show, don't just tell.** A few labelled examples lock an output shape better than a
  paragraph of instructions.
- **Make thinking explicit** (CoT), or **branch it** (ToT) when the first idea is usually wrong.
- **Prohibit clearly.** "Do NOT use X" is obeyed; "keep it clean" is not.
- **Ask it to check itself** - re-derive, test each claim, or admit uncertainty.
- **Give it tools** (ReAct) for anything recent or verifiable.
- **Vote** (self-consistency) and **demand JSON** when reliability matters.

### Exercises
1. Add a `style` axis to the playground: run the same WITH prompt at temperature 0, 0.4, 0.9.
2. Take one of your own past ChatGPT prompts and rewrite it in RCGCOS. Compare across the 3 models.
3. Break the JSON demo: find a review that makes one model emit invalid JSON, then fix the prompt.
4. Build a 2-step prompt chain: prompt A extracts facts, prompt B writes a summary using only those facts.
5. For ReAct, add a second tool (a calculator) and ask a question needing both search and arithmetic.
